# 03_1｜Equipment Random Forest 異常分類訓練

使用專案的 data/processed/fccl_process_deviation_features.csv，以設備參數偏差建立 Random Forest 二元分類模型，目標為 Injected_Anomaly（0＝正常、1＝異常）。

流程包含資料檢查、設備特徵選取、分層切分、交叉驗證參數搜尋、測試集評估、特徵重要度檢視，以及匯出獨立的 models/equipment_anomaly_model.joblib；不會覆寫現有 random_forest_anomaly.joblib。

> **使用限制：** 專案資料是合成資料，異常標籤與設備偏差是教學用假設。模型指標只代表這份合成資料的結果，部署到真實產線前必須以真實資料重新訓練與驗證。


In [ ]:
from pathlib import Path
import warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    balanced_accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42
TARGET_COLUMN = "Injected_Anomaly"
DATA_FILE = "fccl_process_deviation_features.csv"

# 從目前目錄或其上層尋找專案資料，可在 repo 根目錄或 notebooks 資料夾執行。
project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data" / "processed" / DATA_FILE).exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("找不到 data/processed/fccl_process_deviation_features.csv。")

data_path = project_root / "data" / "processed" / DATA_FILE
model_dir = project_root / "models"
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "equipment_anomaly_model.joblib"

# 自動選擇中文字型；沒有中文字型時模型仍可正常訓練。
font_candidates = [
    "Microsoft JhengHei", "Microsoft YaHei", "Noto Sans CJK TC",
    "Noto Sans CJK SC", "PingFang TC", "SimHei",
]
installed_fonts = {font.name for font in plt.matplotlib.font_manager.fontManager.ttflist}
chinese_font = next((name for name in font_candidates if name in installed_fonts), None)
if chinese_font:
    plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
else:
    warnings.warn("找不到中文字型，圖表標籤可能無法正確顯示。")
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid")

print(f"資料來源：{data_path}")
print(f"模型輸出：{model_path}")


## 1. 載入資料與檢查標籤

目標欄位為 Injected_Anomaly。QC_Result 是檢驗結果，Anomaly_Pattern 是事後異常描述；兩者不作為模型特徵，以免直接讀到答案。批次 ID 與日期欄位也排除，避免模型記住識別資訊或時間索引。

數值特徵使用設備參數的絕對偏差與百分比偏差；產線及產品型號作為類別特徵。

In [ ]:
df = pd.read_csv(data_path)
parameters = ["speed", "unwind", "viscosity", "gap", "pump",
              "wet", "temp", "pressure", "cure", "rewind"]
required = {TARGET_COLUMN, "Line", "Product_Model"}
required.update(f"{name}_deviation" for name in parameters)
missing = sorted(required - set(df.columns))
if missing:
    raise ValueError(f"CSV 缺少必要欄位：{missing}")

# 僅選設備偏差欄位；不輸入 QC 結果、日期或批次識別資訊。
categorical_features = ["Line", "Product_Model"]
numeric_features = [
    c for c in df.columns
    if c.endswith("_deviation") or c.endswith("_deviation_pct")
]
feature_columns = categorical_features + numeric_features
excluded_columns = [c for c in df.columns if c not in feature_columns and c != TARGET_COLUMN]
X = df[feature_columns].copy()
y = df[TARGET_COLUMN].astype(int)
if y.nunique() != 2 or set(y.unique()) != {0, 1}:
    raise ValueError("Injected_Anomaly 必須同時包含 0（正常）與 1（異常）。")

print(f"樣本數：{len(df)}；設備數值特徵：{len(numeric_features)}")
display(y.value_counts().sort_index().rename(index={0:"正常（0）",1:"異常（1）"}).to_frame("筆數"))
display(df.groupby(["Line","Product_Model"])[TARGET_COLUMN].agg(["count","sum"]).rename(columns={"count":"批次數","sum":"異常數"}))


## 2. 分層切分與前處理

保留 20% 作為最後評估的測試集，並使用分層抽樣維持正常與異常比例。缺值填補、類別編碼與分類器放在同一個 Pipeline，交叉驗證時每一折只用該折訓練資料學習前處理參數。Random Forest 不需要數值標準化。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
], remainder="drop")
pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=RANDOM_STATE, class_weight="balanced", n_jobs=1
    )),
])
print(f"訓練集：{len(X_train)}；測試集：{len(X_test)}")
print(f"訓練異常比例：{y_train.mean():.1%}；測試異常比例：{y_test.mean():.1%}")


## 3. Random Forest 參數最佳化

使用 5 折分層交叉驗證搭配 RandomizedSearchCV，搜尋樹數量、樹深、葉節點樣本數、特徵抽樣方式、分裂準則及類別權重。以異常類別 F1 為主要指標；獨立測試集不參與調參。

In [ ]:
parameter_distributions = {
    "classifier__n_estimators": [200, 400, 600],
    "classifier__max_depth": [None, 4, 6, 8, 12, 16],
    "classifier__min_samples_split": [2, 4, 6, 10],
    "classifier__min_samples_leaf": [1, 2, 3, 4],
    "classifier__max_features": ["sqrt", 0.5, 1.0],
    "classifier__class_weight": ["balanced", "balanced_subsample", None],
    "classifier__criterion": ["gini", "entropy", "log_loss"],
}
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
search = RandomizedSearchCV(
    estimator=pipeline, param_distributions=parameter_distributions,
    n_iter=30, scoring="f1", cv=cv_strategy, refit=True,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=1, return_train_score=True,
)
search.fit(X_train, y_train)
best_model = search.best_estimator_
best_params = search.best_params_
cv_best_f1 = float(search.best_score_)
print(f"最佳 5 折異常 F1：{cv_best_f1:.4f}")
display(pd.Series(best_params, name="最佳值").to_frame())


## 4. 獨立測試集評估

以下指標使用未參與模型搜尋的測試資料。異常類別是正類（1）；檢視 Precision、Recall、F1、ROC-AUC、PR-AUC、Balanced Accuracy 與混淆矩陣。

In [ ]:
y_pred = best_model.predict(X_test)
class_order = list(best_model.named_steps["classifier"].classes_)
y_probability = best_model.predict_proba(X_test)[:, class_order.index(1)]
test_metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision_anomaly": float(precision_score(y_test, y_pred, pos_label=1, zero_division=0)),
    "recall_anomaly": float(recall_score(y_test, y_pred, pos_label=1, zero_division=0)),
    "f1_anomaly": float(f1_score(y_test, y_pred, pos_label=1, zero_division=0)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_probability)),
    "pr_auc_average_precision": float(average_precision_score(y_test, y_probability)),
}
display(pd.Series(test_metrics, name="測試集指標").to_frame())
print(classification_report(y_test, y_pred, labels=[0,1],
                            target_names=["正常","異常"], zero_division=0))
confusion = confusion_matrix(y_test, y_pred, labels=[0,1])
fig, ax = plt.subplots(figsize=(5.5,4.5))
ConfusionMatrixDisplay(confusion, display_labels=["正常","異常"]).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d"
)
ax.set_title("設備異常分類：測試集混淆矩陣")
plt.tight_layout()
plt.show()


## 5. 特徵重要度

特徵重要度用於查看模型在這份資料中常用哪些欄位。它表示模型的預測依賴程度，不代表設備參數具有因果影響，也不能直接作為調機方向。

In [ ]:
encoded_feature_names = best_model.named_steps["preprocess"].get_feature_names_out()
importance = best_model.named_steps["classifier"].feature_importances_
feature_importance = (
    pd.DataFrame({"特徵": encoded_feature_names, "重要度": importance})
    .sort_values("重要度", ascending=False)
    .reset_index(drop=True)
)
display(feature_importance.head(20))
top_features = feature_importance.head(20).sort_values("重要度")
fig, ax = plt.subplots(figsize=(9,7))
ax.barh(top_features["特徵"], top_features["重要度"], color="#2563eb")
ax.set_title("設備異常分類：前 20 個特徵重要度")
ax.set_xlabel("Random Forest feature importance")
plt.tight_layout()
plt.show()


## 6. 匯出 equipment 模型

匯出 Pipeline、特徵欄位順序、目標定義、最佳參數及測試集指標，供後續推論程式載入。模型獨立存成 models/equipment_anomaly_model.joblib，不會取代儀表板目前載入的分類模型。

> 專案資料是合成資料。真實產線使用前，請以真實設備偏差與確認過的異常標籤重新訓練，並按時間或批次群組保留獨立驗證資料。

In [ ]:
model_bundle = {
    "model": best_model,
    "feature_columns": feature_columns,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "target_column": TARGET_COLUMN,
    "target_mapping": {0: "正常", 1: "異常"},
    "excluded_columns": excluded_columns,
    "best_params": best_params,
    "cv_best_f1": cv_best_f1,
    "test_metrics": test_metrics,
    "random_state": RANDOM_STATE,
    "model_scope": "equipment deviation anomaly classification",
    "data_is_synthetic": True,
}
joblib.dump(model_bundle, model_path)

# 載回模型檔，確認檔案可讀取且特徵欄位順序一致。
saved_bundle = joblib.load(model_path)
assert saved_bundle["feature_columns"] == feature_columns
assert saved_bundle["target_column"] == TARGET_COLUMN
print(f"已匯出：{model_path}")
print(f"模型特徵數：{len(saved_bundle['feature_columns'])}")
print(f"測試集異常 F1：{saved_bundle['test_metrics']['f1_anomaly']:.4f}")
